# Dự đoán `test.json` từ mô hình đã lưu (DKA-AKI)

## 1. Thư viện

In [1]:
import os
import json

import numpy as np
import pandas as pd
import joblib

## 2. Cấu hình — chọn mô hình & ngưỡng

In [2]:
MODEL_NAME = 'randomforest'          # 'randomforest'
THRESHOLD  = 0.47               # ngưỡng quyết định cho nhãn 0/1

DATA_DIR   = os.path.join('..', 'data')
OUTPUT_DIR = os.path.join('..', 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_FILES = {
    'randomforest': 'randomforest_model.joblib',
    'xgboost':      'xgboost_model.joblib',
}
assert MODEL_NAME in MODEL_FILES, 'MODEL_NAME phải là một trong %s' % list(MODEL_FILES)
MODEL_PATH = os.path.join(OUTPUT_DIR, MODEL_FILES[MODEL_NAME])
OUT_PATH   = os.path.join(OUTPUT_DIR, 'test_predictions_%s.csv' % MODEL_NAME)
print('Mô hình :', MODEL_PATH)
print('Kết quả :', OUT_PATH)
print('Ngưỡng  :', THRESHOLD)

Mô hình : ../output/randomforest_model.joblib
Kết quả : ../output/test_predictions_randomforest.csv
Ngưỡng  : 0.47


## 3. Nạp mô hình & bộ tiền xử lý

In [3]:
model = joblib.load(MODEL_PATH)
print('Đã nạp mô hình:', type(model).__name__)

art = joblib.load(os.path.join(DATA_DIR, 'preprocessor.joblib'))
imputer      = art['imputer']
scaler       = art['scaler']
numeric_cols = art['numeric_cols']
feature_cols = art['feature_cols']
dropped_cols = art['dropped_cols']
TS_KEYS      = art['ts_keys']
STATS        = art['stats']
print('Số đặc trưng mô hình kỳ vọng:', len(feature_cols))

Đã nạp mô hình: RandomForestClassifier
Số đặc trưng mô hình kỳ vọng: 77


## 4. Hàm tiền xử lý (giống quy trình train)

In [4]:
# Các nhóm cột (giống hệt preprocessing.ipynb / modeling_*.ipynb)
NUM_SCALAR = ['age', 'oasis', 'saps2', 'sofa', 'preiculos', 'egfr']
ALWAYS_BOOL = ['chronic_pulmonary_disease', 'congestive_heart_failure', 'malignant_cancer']
COMORBID_FLAGS = ['hypertension', 'microangiopathy', 'macroangiopathy',
                  'history_ami', 'uti', 'history_aci']
CKD = 'ckd_stage'
ID_COLS = ['subjectId', 'hadmId', 'stayId']


def agg_series(ts):
    a = np.array(list(ts.values()), dtype=float)
    return np.nanmin(a), np.nanmean(a), np.nanmax(a)


def flatten(record):
    m = record['measures']
    row = {'subjectId': record['subjectId'], 'hadmId': record['hadmId'], 'stayId': record['stayId']}
    for k in TS_KEYS:
        v = m.get(k)
        vals = agg_series(v) if isinstance(v, dict) and len(v) > 0 else (np.nan, np.nan, np.nan)
        for s, val in zip(STATS, vals):
            row[f'{k}_{s}'] = val
    for k in NUM_SCALAR + ['gender', 'race', 'liver_disease', 'dka_type']:
        row[k] = m.get(k, np.nan)
    for k in ALWAYS_BOOL + COMORBID_FLAGS:
        row[k] = int(bool(m.get(k, False)))
    row[CKD] = m.get(CKD, 0)
    return row


def group_race(s):
    s = str(s).upper()
    if 'WHITE' in s:
        return 'White'
    if 'BLACK' in s or 'AFRICAN' in s:
        return 'Black'
    if 'HISPANIC' in s or 'LATINO' in s:
        return 'Hispanic'
    if 'ASIAN' in s:
        return 'Asian'
    return 'Other'


def preprocess_records(records):
    """Biến đổi danh sách bản ghi -> (bảng id, ma trận đặc trưng X) khớp với train."""
    df = pd.DataFrame([flatten(r) for r in records])
    ids = df[ID_COLS].copy()
    df = df.drop(columns=dropped_cols, errors='ignore')
    df[numeric_cols] = imputer.transform(df[numeric_cols])
    df['gender'] = df['gender'].map({'F': 0, 'M': 1}).fillna(0).astype(int)
    df['liver_disease'] = df['liver_disease'].map({'NONE': 0, 'MILD': 1, 'SEVERE': 2}).fillna(0).astype(int)
    df['race_group'] = df['race'].apply(group_race)
    df = df.drop(columns=['race'])
    df['dka_type'] = df['dka_type'].map({1: 'T1DM', 2: 'T2DM', 0: 'Other'}).fillna('Other')
    df = pd.get_dummies(df, columns=['race_group', 'dka_type'], prefix=['race', 'dka'])
    df[df.select_dtypes(include='bool').columns] = df.select_dtypes(include='bool').astype(int)
    df[numeric_cols] = scaler.transform(df[numeric_cols])
    X_out = df.reindex(columns=feature_cols, fill_value=0)
    return ids, X_out

## 5. Đọc & tiền xử lý `test.json`

In [5]:
with open(os.path.join(DATA_DIR, 'test.json'), encoding='utf-8') as f:
    test = json.load(f)

ids_test, X_test = preprocess_records(test)
print('Số bản ghi test     :', len(test))
print('Kích thước X_test   :', X_test.shape)
print('Khớp đúng cột train?:', list(X_test.columns) == list(feature_cols))
print('Còn giá trị thiếu?  :', bool(X_test.isnull().any().any()))

Số bản ghi test     : 243
Kích thước X_test   : (243, 77)
Khớp đúng cột train?: True
Còn giá trị thiếu?  : False


## 6. Dự đoán & lưu kết quả ra `output/`

File CSV gồm đúng 3 cột: `id` (= `subjectId`), `probability`, `prediction` (ngưỡng `THRESHOLD`).
Mỗi `subjectId` chỉ giữ **một** bản ghi (đầu tiên), giống các notebook huấn luyện.

In [6]:
proba_test = model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= THRESHOLD).astype(int)

result = pd.DataFrame({
    'id': ids_test['subjectId'].values,
    'probability': np.round(proba_test, 4),
    'prediction': pred_test,
})
n_before = len(result)
result = result.drop_duplicates(subset='id', keep='first').reset_index(drop=True)
print('Trước khi gộp: {} | sau khi gộp theo id: {} ({} trùng đã loại)'.format(
    n_before, len(result), n_before - len(result)))

result.to_csv(OUT_PATH, index=False)
print('Đã lưu:', OUT_PATH, '| kích thước:', result.shape)
print('Số ca dự đoán AKI: {} / {} ({:.1%})'.format(
    int(result['prediction'].sum()), len(result), result['prediction'].mean()))
result.head()

Trước khi gộp: 243 | sau khi gộp theo id: 222 (21 trùng đã loại)
Đã lưu: ../output/test_predictions_randomforest.csv | kích thước: (222, 3)
Số ca dự đoán AKI: 104 / 222 (46.8%)


,id,probability,prediction
0,12789108,0.1127,0
1,11569093,0.4985,1
2,17405009,0.1637,0
3,11721696,0.7015,1
4,19822093,0.5633,1
